In [ ]:
"""
Project 03: Customer Segmentation & CLV
Step 02: Segmentation & CLV Model Training
"""

In [ ]:
import warnings

In [ ]:
warnings.filterwarnings("ignore")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib

In [ ]:
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pickle
import os
import time
from tqdm import tqdm
from sklearn.preprocessing import StandardScaler, PowerTransformer
from sklearn.cluster import KMeans, DBSCAN
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
from sklearn.metrics import silhouette_score, mean_absolute_error, r2_score

In [ ]:
BASE = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
RAW = os.path.join(BASE, "data", "raw")
PROCESSED = os.path.join(BASE, "data", "processed")
MODELS = os.path.join(BASE, "models")
CHARTS = os.path.join(BASE, "charts")

In [ ]:
os.makedirs(PROCESSED, exist_ok=True)
os.makedirs(MODELS, exist_ok=True)
os.makedirs(CHARTS, exist_ok=True)

In [ ]:
print("=" * 60)
print("STEP 02: Segmentation & CLV Model Training")
print("=" * 60)
start_time = time.time()

In [ ]:
# ── 1. Load Processed Data ──
proc_path = os.path.join(PROCESSED, "rfm_clv_data.csv")
df = pd.read_csv(proc_path, index_col=0)
print(f"Loaded RFM data: {df.shape}")
print(df.head())

In [ ]:
# ── 2. Data Transformation ──
print("\nTransforming RFM features...")
features = ["Recency", "Frequency", "Monetary"]
X_cl = df[features]

In [ ]:
# Yeo-Johnson Power Transform + StandardScaler
pt = PowerTransformer(method="yeo-johnson")
scaler = StandardScaler()

In [ ]:
X_transformed = pt.fit_transform(X_cl)
X_scaled = scaler.fit_transform(X_transformed)

In [ ]:
with open(os.path.join(MODELS, "transformer.pkl"), "wb") as f:
    pickle.dump(pt, f)
with open(os.path.join(MODELS, "scaler.pkl"), "wb") as f:
    pickle.dump(scaler, f)
print("Saved: transformer.pkl, scaler.pkl")

In [ ]:
# ── 3. K-Means Clustering ──
print("\nTraining K-Means (k=4)...")
k_range = range(2, 9)
sil_scores = []

In [ ]:
for k in tqdm(k_range, desc="Testing K values"):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    sil_scores.append(silhouette_score(X_scaled, labels))

In [ ]:
best_k = list(k_range)[np.argmax(sil_scores)]
print(f"Best k by silhouette: {best_k} (score: {max(sil_scores):.4f})")

In [ ]:
# Train final K-Means
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df["Cluster"] = kmeans.fit_predict(X_scaled)

In [ ]:
sil_score = silhouette_score(X_scaled, df["Cluster"])
print(f"K-Means (k=4) Silhouette Score: {sil_score:.4f}")

In [ ]:
with open(os.path.join(MODELS, "kmeans_model.pkl"), "wb") as f:
    pickle.dump(kmeans, f)
print("Saved: kmeans_model.pkl")

In [ ]:
# Elbow/Silhouette chart
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(list(k_range), sil_scores, "bo-", linewidth=2, markersize=8)
axes[0].axvline(x=4, color="red", linestyle="--", alpha=0.7, label="Selected k=4")
axes[0].set_title("Silhouette Score vs K", fontsize=14, fontweight="bold")
axes[0].set_xlabel("Number of Clusters (K)")
axes[0].set_ylabel("Silhouette Score")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

In [ ]:
# Inertia
inertias = []
for k in tqdm(k_range, desc="Computing inertia"):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

In [ ]:
axes[1].plot(list(k_range), inertias, "go-", linewidth=2, markersize=8)
axes[1].axvline(x=4, color="red", linestyle="--", alpha=0.7, label="Selected k=4")
axes[1].set_title("Elbow Method (Inertia)", fontsize=14, fontweight="bold")
axes[1].set_xlabel("Number of Clusters (K)")
axes[1].set_ylabel("Inertia")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

In [ ]:
plt.tight_layout()
plt.savefig(os.path.join(CHARTS, "cluster_selection.png"), dpi=150, bbox_inches="tight")
plt.close()
print("Saved: cluster_selection.png")

In [ ]:
# ── 4. DBSCAN Outlier Detection ──
print("\nRunning DBSCAN for VIP outlier detection...")
dbscan = DBSCAN(eps=0.5, min_samples=5)
df["DBSCAN_Outlier"] = dbscan.fit_predict(X_scaled)

In [ ]:
vip_count = len(df[df["DBSCAN_Outlier"] == -1])
print(f"DBSCAN detected {vip_count} potential VIP outliers.")

In [ ]:
with open(os.path.join(MODELS, "dbscan_model.pkl"), "wb") as f:
    pickle.dump(dbscan, f)
print("Saved: dbscan_model.pkl")

In [ ]:
# ── 5. Cluster Profile Analysis ──
print("\nCluster profiles:")
cluster_names = {0: "Champions", 1: "At-Risk", 2: "Loyal", 3: "About to Sleep"}
cluster_profile = df.groupby("Cluster")[features + ["CLV_Target"]].mean().round(2)
print(cluster_profile)

In [ ]:
# Cluster distribution chart
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, feat in enumerate(tqdm(features, desc="Plotting cluster profiles")):
    for c in range(4):
        data = df[df["Cluster"] == c][feat]
        axes[i].hist(data, bins=30, alpha=0.5, label=f"Cluster {c}", edgecolor="white")
    axes[i].set_title(f"{feat} by Cluster", fontsize=13, fontweight="bold")
    axes[i].set_xlabel(feat)
    axes[i].set_ylabel("Count")
    axes[i].legend()
    axes[i].set_yscale("log")

In [ ]:
plt.tight_layout()
plt.savefig(os.path.join(CHARTS, "cluster_profiles.png"), dpi=150, bbox_inches="tight")
plt.close()
print("Saved: cluster_profiles.png")

In [ ]:
# Cluster size pie chart
fig, ax = plt.subplots(figsize=(8, 8))
cluster_sizes = df["Cluster"].value_counts().sort_index()
colors = ["#3b82f6", "#ef4444", "#10b981", "#f59e0b"]
labels = [cluster_names.get(i, f"Cluster {i}") for i in cluster_sizes.index]
ax.pie(
    cluster_sizes.values,
    labels=labels,
    colors=colors,
    autopct="%1.1f%%",
    startangle=90,
    textprops={"fontsize": 12},
)
ax.set_title("Customer Segment Distribution", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(
    os.path.join(CHARTS, "cluster_distribution.png"), dpi=150, bbox_inches="tight"
)
plt.close()
print("Saved: cluster_distribution.png")

In [ ]:
# ── 6. Supervised CLV Regression ──
print("\nTraining CLV regression models...")
X_reg = np.column_stack([X_scaled, df["Cluster"]])
y_reg = df["CLV_Target"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)
print(f"Train: {X_train.shape[0]}, Test: {X_test.shape[0]}")

In [ ]:
model_configs = {
    "ridge": Ridge(alpha=1.0),
    "rf": RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42),
    "gb": GradientBoostingRegressor(
        n_estimators=100, learning_rate=0.1, random_state=42
    ),
}

In [ ]:
results = []
for name, model in tqdm(model_configs.items(), desc="Training models"):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    results.append({"Model": name.upper(), "MAE": mae, "R2": r2})
    print(f"  {name.upper():10s} | MAE: {mae:10.2f} | R2: {r2:.4f}")

    with open(os.path.join(MODELS, f"{name}_clv_model.pkl"), "wb") as f:
        pickle.dump(model, f)

In [ ]:
# Save results
results_df = pd.DataFrame(results)
results_df.to_csv(os.path.join(MODELS, "clv_model_results.csv"), index=False)
print("\nSaved: clv_model_results.csv")

In [ ]:
# Model comparison chart
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
model_names = results_df["Model"].values
colors = ["#3b82f6", "#10b981", "#f59e0b"]

In [ ]:
axes[0].bar(
    model_names, results_df["MAE"], color=colors, edgecolor="white", linewidth=2
)
axes[0].set_title("MAE Comparison (Lower = Better)", fontsize=14, fontweight="bold")
axes[0].set_ylabel("Mean Absolute Error")
for i, v in enumerate(results_df["MAE"]):
    axes[0].text(i, v + 20, f"${v:.0f}", ha="center", fontweight="bold")

In [ ]:
axes[1].bar(model_names, results_df["R2"], color=colors, edgecolor="white", linewidth=2)
axes[1].set_title(
    "R2 Score Comparison (Higher = Better)", fontsize=14, fontweight="bold"
)
axes[1].set_ylabel("R2 Score")
for i, v in enumerate(results_df["R2"]):
    axes[1].text(i, v + 0.01, f"{v:.4f}", ha="center", fontweight="bold")

In [ ]:
plt.tight_layout()
plt.savefig(os.path.join(CHARTS, "model_comparison.png"), dpi=150, bbox_inches="tight")
plt.close()
print("Saved: model_comparison.png")

In [ ]:
# Save updated data with clusters
df.to_csv(proc_path)
print(f"Updated processed data saved.")

In [ ]:
elapsed = time.time() - start_time
print(f"\n{'=' * 60}")
print(f"STEP 02 COMPLETE | Time: {elapsed:.1f}s")
print(f"  K-Means Silhouette: {sil_score:.4f}")
print(
    f"  Best CLV Model: {results_df.loc[results_df['R2'].idxmax(), 'Model']} (R2={results_df['R2'].max():.4f})"
)
print(f"{'=' * 60}")